In [ ]:

# ================================================================
# PROJETO DE ZOONOSES
# GERADOR DE DADOS BRUTOS — SQLITE
# Versão revisada
#
# - Geração inicial e incremental.
# - Preservação dos lotes anteriores.
# - Identificadores técnicos sequenciais.
# - Problemas controlados para tratamento.
# - Duplicidades exatas e parciais.
# - Relacionamentos por identificador de origem.
# - Vacinas compatíveis com cada espécie.
# ================================================================


# 1. IMPORTAÇÕES E CONFIGURAÇÕES

import os
import re
import json
import random
import sqlite3
import math

from copy import deepcopy
from datetime import date, datetime, timedelta


PASTA_PROJETO = "/content/PROJETO_ZOONOSES"

CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

# Quantidade de animais originais deste lote.
QUANTIDADE_ANIMAIS = 5000

# Percentual de problemas introduzidos nos registros originais.
TAXA_PROBLEMAS = 0.12

# Percentual de duplicidades adicionais de animais.
TAXA_DUPLICADOS = 0.02

# INICIAL: primeira geração de dados brutos.
# ADICIONAR: acrescenta um novo lote sem apagar os anteriores.
MODO_GERACAO = "INICIAL"


# 2. VALIDAÇÕES INICIAIS

if not os.path.isdir(PASTA_PROJETO):
    raise FileNotFoundError(
        "Pasta do projeto não encontrada. "
        "Execute primeiro o notebook do banco de dados."
    )

if not os.path.exists(CAMINHO_BANCO):
    raise FileNotFoundError(
        "Banco SQLite não encontrado. "
        "Confira o caminho do arquivo."
    )

if (
    not isinstance(QUANTIDADE_ANIMAIS, int)
    or isinstance(QUANTIDADE_ANIMAIS, bool)
    or QUANTIDADE_ANIMAIS <= 0
):
    raise ValueError(
        "QUANTIDADE_ANIMAIS precisa ser um inteiro maior que zero."
    )

if not 0 <= TAXA_PROBLEMAS <= 1:
    raise ValueError(
        "TAXA_PROBLEMAS deve estar entre 0 e 1."
    )

if not 0 <= TAXA_DUPLICADOS <= 1:
    raise ValueError(
        "TAXA_DUPLICADOS deve estar entre 0 e 1."
    )

if MODO_GERACAO not in {"INICIAL", "ADICIONAR"}:
    raise ValueError(
        "MODO_GERACAO deve ser 'INICIAL' ou 'ADICIONAR'."
    )


# 3. CONEXÃO COM O SQLITE

conexao = sqlite3.connect(
    CAMINHO_BANCO,
    timeout=60
)

conexao.execute("PRAGMA foreign_keys = ON")
conexao.execute("PRAGMA busy_timeout = 60000")

cursor = conexao.cursor()


# 4. VERIFICAÇÃO DAS TABELAS E COLUNAS

tabelas_necessarias = {
    "lotes",
    "registros_brutos",
    "auditoria"
}

tabelas_existentes = {
    linha[0]
    for linha in cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
    """).fetchall()
}

faltantes = tabelas_necessarias - tabelas_existentes

if faltantes:
    conexao.close()

    raise Exception(
        "Tabelas necessárias não encontradas: "
        + ", ".join(sorted(faltantes))
    )


colunas_brutos = {
    linha[1]
    for linha in cursor.execute(
        "PRAGMA table_info(registros_brutos)"
    ).fetchall()
}

colunas_brutos_necessarias = {
    "id_lote",
    "conjunto",
    "dados_json",
    "data_recebimento"
}

if not colunas_brutos_necessarias.issubset(colunas_brutos):
    conexao.close()

    raise Exception(
        "A tabela registros_brutos não possui todas "
        "as colunas necessárias."
    )


# 5. VERIFICAÇÃO DO MODO DE GERAÇÃO

quantidade_animais_anteriores = cursor.execute("""
    SELECT COUNT(*)
    FROM registros_brutos
    WHERE conjunto = 'animais'
""").fetchone()[0]


if (
    MODO_GERACAO == "INICIAL"
    and quantidade_animais_anteriores > 0
):
    conexao.close()

    raise Exception(
        "Já existem animais brutos neste banco. "
        "Use MODO_GERACAO = 'ADICIONAR' "
        "para acrescentar outro lote."
    )


# 6. CONTROLE DA SEQUÊNCIA DE IDENTIFICADORES

cursor.execute("""
CREATE TABLE IF NOT EXISTS controle_sequencias (
    nome TEXT PRIMARY KEY,
    proximo_valor INTEGER NOT NULL
)
""")

cursor.execute("""
INSERT OR IGNORE INTO controle_sequencias
(nome, proximo_valor)
VALUES ('animais', 1)
""")

conexao.commit()


# Recupera a sequência registrada.
proximo_numero = cursor.execute("""
    SELECT proximo_valor
    FROM controle_sequencias
    WHERE nome = 'animais'
""").fetchone()[0]


# Confere também os identificadores já existentes.
# Isso evita repetir IDs caso a tabela de sequência
# tenha sido criada depois de dados antigos.

maior_numero_existente = 0


def atualizar_maior_numero(identificador):

    global maior_numero_existente

    if not identificador:
        return

    correspondencia = re.fullmatch(
        r"ANM(\d+)",
        str(identificador)
    )

    if correspondencia:

        numero = int(correspondencia.group(1))

        maior_numero_existente = max(
            maior_numero_existente,
            numero
        )


# Verifica animais já tratados.
if "animais" in tabelas_existentes:

    colunas_animais = {
        linha[1]
        for linha in cursor.execute(
            "PRAGMA table_info(animais)"
        ).fetchall()
    }

    if "id_animal" in colunas_animais:

        for linha in cursor.execute("""
            SELECT id_animal
            FROM animais
        """).fetchall():

            atualizar_maior_numero(linha[0])


# Verifica os identificadores dos registros brutos.
for linha in cursor.execute("""
    SELECT dados_json
    FROM registros_brutos
    WHERE conjunto = 'animais'
""").fetchall():

    try:

        registro = json.loads(linha[0])

        atualizar_maior_numero(
            registro.get("id_animal")
        )

        meta = registro.get("_meta", {})

        atualizar_maior_numero(
            meta.get("id_animal_origem")
        )

    except (json.JSONDecodeError, TypeError):
        continue


proximo_numero = max(
    proximo_numero,
    maior_numero_existente + 1
)


# 7. REFERÊNCIAS DOS DADOS SINTÉTICOS

municipios = [
    "Feijó",
    "Cruzeiro do Sul",
    "Sena Madureira",
    "Rio Branco",
    "Tarauacá"
]

especies = [
    "Cachorro",
    "Gato"
]

nomes_cachorros = [
    "Rex", "Thor", "Bob", "Max", "Toby", "Bidu",
    "Pingo", "Fred", "Zeus", "Billy", "Scooby",
    "Bolinha", "Tito", "Apolo", "Bento", "Nino",
    "Marley", "Simba", "Lucky", "Pipoca",
    "Caramelo", "Duque", "Costelinha", "Toddy",
    "Paçoca", "Pudim", "Fubá", "Boris"
]

nomes_gatos = [
    "Mimi", "Luna", "Nina", "Mel", "Mingau",
    "Frajola", "Tom", "Garfield", "Chico",
    "Pérola", "Amora", "Salem", "Fumaça",
    "Pipoca", "Biscoito", "Tigrinho", "Nala",
    "Simba", "Mittens", "Félix", "Pudim",
    "Bolinha", "Manchinha", "Cacau", "Floquinho"
]

racas_cachorros = [
    "Sem raça definida",
    "Pinscher",
    "Poodle",
    "Labrador",
    "Golden Retriever",
    "Pastor Alemão",
    "Shih-tzu",
    "Dachshund",
    "Pit Bull",
    "Beagle",
    "Border Collie",
    "Rottweiler"
]

racas_gatos = [
    "Sem raça definida",
    "Siamês",
    "Persa",
    "Maine Coon",
    "Angorá",
    "Bengal",
    "Ragdoll"
]

motivos_acolhimento = [
    "Maus-tratos",
    "Abandono",
    "Acidente",
    "Doença",
    "Recolhimento preventivo",
    "Entrega voluntária",
    "Animal perdido",
    "Resgate de filhotes",
    "Outros"
]

origens_acolhimento = [
    "Resgate pela equipe",
    "Denúncia da população",
    "Entrega pelo tutor",
    "Órgão público",
    "Clínica veterinária",
    "Recolhimento nas ruas"
]

# Vacinas separadas por espécie.
vacinas_por_especie = {
    "Cachorro": [
        "Antirrábica",
        "V8",
        "V10"
    ],

    "Gato": [
        "Antirrábica",
        "Polivalente felina",
        "Leucemia felina"
    ]
}

diagnosticos_disponiveis = [
    "Leishmaniose",
    "Esporotricose",
    "Erliquiose",
    "Cinomose",
    "Parvovirose",
    "Dermatite",
    "Sarna",
    "Verminose",
    "Infecção respiratória",
    "Gastroenterite"
]

tipos_maus_tratos = [
    "Abandono",
    "Agressão física",
    "Negligência",
    "Falta de alimentação",
    "Falta de água",
    "Confinamento inadequado",
    "Ausência de atendimento veterinário"
]

motivos_obito = [
    "Doença",
    "Acidente",
    "Complicações clínicas",
    "Causas naturais"
]

justificativas_eutanasia = [
    "Sofrimento irreversível",
    "Doença em estágio terminal",
    "Condição clínica sem possibilidade de recuperação"
]


# 8. FUNÇÕES AUXILIARES

def data_aleatoria(inicio="2022-01-01", fim=None):

    if fim is None:
        fim = date.today().isoformat()

    inicio_data = datetime.strptime(
        inicio,
        "%Y-%m-%d"
    ).date()

    fim_data = datetime.strptime(
        fim,
        "%Y-%m-%d"
    ).date()

    if inicio_data > fim_data:
        raise ValueError(
            "A data inicial não pode ser posterior à data final."
        )

    diferenca = (fim_data - inicio_data).days

    return (
        inicio_data
        + timedelta(days=random.randint(0, diferenca))
    ).isoformat()


def data_futura(dias=90):

    return (
        date.today()
        + timedelta(days=random.randint(1, dias))
    ).isoformat()


def peso_aleatorio(especie, raca):

    if especie == "Gato":
        return round(random.uniform(2.0, 8.0), 2)

    if raca == "Pinscher":
        return round(random.uniform(2.0, 6.0), 2)

    if raca == "Dachshund":
        return round(random.uniform(4.0, 12.0), 2)

    if raca in [
        "Labrador",
        "Golden Retriever",
        "Pastor Alemão",
        "Rottweiler"
    ]:
        return round(random.uniform(18.0, 38.0), 2)

    if raca == "Pit Bull":
        return round(random.uniform(15.0, 35.0), 2)

    return round(random.uniform(3.0, 25.0), 2)


def criar_meta(id_animal_origem=None):

    return {
        "id_animal_origem": id_animal_origem,
        "problemas_introduzidos": []
    }


def registrar_problema(registro, problema):

    registro.setdefault("_meta", {})

    registro["_meta"].setdefault(
        "problemas_introduzidos",
        []
    )

    registro["_meta"]["problemas_introduzidos"].append(
        problema
    )


# 9. GERAÇÃO DE UM ANIMAL

def gerar_animal(numero):

    especie = random.choice(especies)

    if especie == "Cachorro":

        nome = random.choice(nomes_cachorros)
        raca = random.choice(racas_cachorros)

    else:

        nome = random.choice(nomes_gatos)
        raca = random.choice(racas_gatos)

    municipio = random.choice(municipios)

    identificador = f"ANM{numero:08d}"

    animal = {
        "id_animal": identificador,
        "nome": nome,
        "especie": especie,
        "raca": raca,
        "sexo": random.choice(["Macho", "Fêmea"]),
        "municipio": municipio,
        "peso_kg": peso_aleatorio(especie, raca),
        "status": "Ativo",
        "vacinas_faltantes": "",
        "situacao_vacinacao": random.choice([
            "Em dia",
            "Pendente",
            "Parcial"
        ]),
        "imagem": (
            f"{municipio.replace(' ', '_')}"
            f"_animal_{numero:08d}.jpg"
        ),
        "data_acolhimento": data_aleatoria(),
        "motivo_acolhimento": random.choice(
            motivos_acolhimento
        ),
        "origem_acolhimento": random.choice(
            origens_acolhimento
        ),
        "_meta": criar_meta(identificador)
    }

    return animal


# 10. GERAÇÃO DAS VACINAÇÕES

def gerar_vacinacoes(animal):

    registros = []

    quantidade = random.choices(
        [0, 1, 2, 3],
        weights=[15, 30, 35, 20]
    )[0]

    data_acolhimento = animal["data_acolhimento"]

    for _ in range(quantidade):

        data_aplicacao = data_aleatoria(
            inicio=data_acolhimento
        )

        proxima_dose = (
            datetime.strptime(
                data_aplicacao,
                "%Y-%m-%d"
            ).date()
            + timedelta(
                days=random.choice([30, 60, 90, 365])
            )
        ).isoformat()

        registros.append({
            "id_animal": animal["id_animal"],
            "nome_vacina": random.choice(
                vacinas_por_especie[animal["especie"]]
            ),
            "data_aplicacao": data_aplicacao,
            "proxima_dose": proxima_dose,
            "situacao": random.choice([
                "Aplicada",
                "Pendente",
                "Aguardando reforço"
            ]),
            "observacoes": (
                "Registro sintético de vacinação."
            ),
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 11. GERAÇÃO DOS DIAGNÓSTICOS

def gerar_diagnosticos(animal):

    registros = []

    if random.random() < 0.12:

        registros.append({
            "id_animal": animal["id_animal"],
            "data_diagnostico": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "diagnostico": random.choice(
                diagnosticos_disponiveis
            ),
            "descricao": (
                "Diagnóstico sintético para fins acadêmicos."
            ),
            "situacao": random.choice([
                "Em tratamento",
                "Em acompanhamento",
                "Recuperado"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 12. GERAÇÃO DAS OCORRÊNCIAS DE MAUS-TRATOS

def gerar_maus_tratos(animal):

    registros = []

    motivo = animal["motivo_acolhimento"]

    gerar_ocorrencia = (
        motivo == "Maus-tratos"
        or random.random() < 0.025
    )

    if gerar_ocorrencia:

        registros.append({
            "id_animal": animal["id_animal"],
            "municipio": animal["municipio"],
            "data_ocorrencia": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "tipo_ocorrencia": random.choice(
                tipos_maus_tratos
            ),
            "descricao": (
                "Ocorrência sintética para análise."
            ),
            "situacao": random.choice([
                "Em investigação",
                "Em acompanhamento",
                "Concluída"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 13. GERAÇÃO DE ÓBITOS E EUTANÁSIA

def gerar_eventos(animal):

    registros = []

    if random.random() < 0.015:

        tipo = random.choices(
            ["Óbito", "Eutanásia"],
            weights=[85, 15]
        )[0]

        if tipo == "Óbito":

            motivo = random.choice(motivos_obito)
            observacoes = "Evento sintético de óbito."

        else:

            motivo = random.choice(
                justificativas_eutanasia
            )

            observacoes = "Evento sintético de eutanásia."

        data_evento = data_aleatoria(
            inicio=animal["data_acolhimento"]
        )

        animal["status"] = "Inativo"

        registros.append({
            "id_animal": animal["id_animal"],
            "tipo_evento": tipo,
            "data_evento": data_evento,
            "municipio": animal["municipio"],
            "motivo": motivo,
            "observacoes": observacoes,
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 14. INTRODUÇÃO CONTROLADA DE PROBLEMAS

def aplicar_problemas(registros, tipo, taxa):

    if not registros:
        return 0

    quantidade = math.ceil(
        len(registros) * taxa
    )

    selecionados = random.sample(
        range(len(registros)),
        min(quantidade, len(registros))
    )

    for indice in selecionados:

        registro = registros[indice]

        if tipo == "animais":

            problema = random.choice([
                "nome_nulo",
                "raca_vazia",
                "peso_negativo",
                "peso_texto",
                "municipio_invalido",
                "sexo_invalido",
                "status_invalido",
                "situacao_vacinacao_invalida",
                "data_acolhimento_futura",
                "motivo_acolhimento_invalido",
                "origem_acolhimento_invalida",
                "id_animal_ausente",
                "id_animal_malformado"
            ])

            if problema == "nome_nulo":
                registro["nome"] = None

            elif problema == "raca_vazia":
                registro["raca"] = ""

            elif problema == "peso_negativo":
                registro["peso_kg"] = -random.uniform(1, 20)

            elif problema == "peso_texto":
                registro["peso_kg"] = "peso desconhecido"

            elif problema == "municipio_invalido":
                registro["municipio"] = "Município inexistente"

            elif problema == "sexo_invalido":
                registro["sexo"] = "Não identificado??"

            elif problema == "status_invalido":
                registro["status"] = "Desconhecido"

            elif problema == "situacao_vacinacao_invalida":
                registro["situacao_vacinacao"] = (
                    "Vacinação incorreta"
                )

            elif problema == "data_acolhimento_futura":
                registro["data_acolhimento"] = data_futura()

            elif problema == "motivo_acolhimento_invalido":
                registro["motivo_acolhimento"] = (
                    "Motivo não cadastrado"
                )

            elif problema == "origem_acolhimento_invalida":
                registro["origem_acolhimento"] = (
                    "Origem desconhecida"
                )

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            elif problema == "id_animal_malformado":
                registro["id_animal"] = "ID-ANIMAL-ERRO"

            registrar_problema(
                registro,
                problema
            )

        elif tipo == "vacinacoes":

            problema = random.choice([
                "vacina_nula",
                "data_aplicacao_futura",
                "proxima_dose_anterior",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "vacina_nula":
                registro["nome_vacina"] = None

            elif problema == "data_aplicacao_futura":
                registro["data_aplicacao"] = data_futura()

            elif problema == "proxima_dose_anterior":
                registro["proxima_dose"] = "2020-01-01"

            elif problema == "situacao_invalida":
                registro["situacao"] = (
                    "Situação desconhecida"
                )

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(
                registro,
                problema
            )

        elif tipo == "diagnosticos":

            problema = random.choice([
                "diagnostico_vazio",
                "data_diagnostico_futura",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "diagnostico_vazio":
                registro["diagnostico"] = ""

            elif problema == "data_diagnostico_futura":
                registro["data_diagnostico"] = data_futura()

            elif problema == "situacao_invalida":
                registro["situacao"] = (
                    "Situação incorreta"
                )

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(
                registro,
                problema
            )

        elif tipo == "maus_tratos":

            problema = random.choice([
                "municipio_invalido",
                "tipo_ocorrencia_vazio",
                "data_ocorrencia_futura",
                "situacao_invalida",
                "id_animal_ausente"
            ])

            if problema == "municipio_invalido":
                registro["municipio"] = "Local desconhecido"

            elif problema == "tipo_ocorrencia_vazio":
                registro["tipo_ocorrencia"] = ""

            elif problema == "data_ocorrencia_futura":
                registro["data_ocorrencia"] = data_futura()

            elif problema == "situacao_invalida":
                registro["situacao"] = (
                    "Situação não cadastrada"
                )

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(
                registro,
                problema
            )

        elif tipo == "obitos_eutanasias":

            problema = random.choice([
                "motivo_ausente",
                "data_evento_anterior_ao_acolhimento",
                "municipio_invalido",
                "tipo_evento_invalido",
                "id_animal_ausente"
            ])

            if problema == "motivo_ausente":
                registro["motivo"] = None

            elif problema == "data_evento_anterior_ao_acolhimento":
                registro["data_evento"] = "2020-01-01"

            elif problema == "municipio_invalido":
                registro["municipio"] = (
                    "Município inexistente"
                )

            elif problema == "tipo_evento_invalido":
                registro["tipo_evento"] = (
                    "Evento desconhecido"
                )

            elif problema == "id_animal_ausente":
                registro["id_animal"] = None

            registrar_problema(
                registro,
                problema
            )

    return len(selecionados)


# 15. GERAÇÃO DOS ANIMAIS E CONJUNTOS RELACIONADOS

print("Iniciando geração dos dados sintéticos...")

animais = []
vacinacoes = []
diagnosticos = []
maus_tratos = []
obitos_eutanasias = []

for i in range(QUANTIDADE_ANIMAIS):

    numero = proximo_numero + i

    animal = gerar_animal(numero)

    animais.append(animal)

    vacinacoes.extend(
        gerar_vacinacoes(animal)
    )

    diagnosticos.extend(
        gerar_diagnosticos(animal)
    )

    maus_tratos.extend(
        gerar_maus_tratos(animal)
    )

    obitos_eutanasias.extend(
        gerar_eventos(animal)
    )

    if (i + 1) % 1000 == 0:
        print(f"{i + 1} animais gerados...")


# 16. INTRODUÇÃO DE INCONSISTÊNCIAS DE STATUS

# Alguns animais com evento terminal serão marcados
# incorretamente como ativos.
#
# O tratamento deverá identificar essa inconsistência
# e corrigir o status para Inativo.

animais_com_evento = {
    evento["_meta"]["id_animal_origem"]
    for evento in obitos_eutanasias
}

animais_por_origem = {
    animal["_meta"]["id_animal_origem"]: animal
    for animal in animais
}

if animais_com_evento:

    candidatos = [
        animais_por_origem[id_origem]
        for id_origem in animais_com_evento
        if id_origem in animais_por_origem
    ]

    quantidade_status_incorreto = math.ceil(
        len(candidatos) * 0.10
    )

    selecionados = random.sample(
        candidatos,
        min(
            quantidade_status_incorreto,
            len(candidatos)
        )
    )

    for animal in selecionados:

        animal["status"] = "Ativo"

        registrar_problema(
            animal,
            "animal_ativo_apos_obito_ou_eutanasia"
        )


# 17. APLICAÇÃO DOS PROBLEMAS POR CONJUNTO

relatorio_problemas = {}

relatorio_problemas["animais"] = aplicar_problemas(
    animais,
    "animais",
    TAXA_PROBLEMAS
)

relatorio_problemas["vacinacoes"] = aplicar_problemas(
    vacinacoes,
    "vacinacoes",
    TAXA_PROBLEMAS
)

relatorio_problemas["diagnosticos"] = aplicar_problemas(
    diagnosticos,
    "diagnosticos",
    TAXA_PROBLEMAS
)

relatorio_problemas["maus_tratos"] = aplicar_problemas(
    maus_tratos,
    "maus_tratos",
    TAXA_PROBLEMAS
)

relatorio_problemas["obitos_eutanasias"] = aplicar_problemas(
    obitos_eutanasias,
    "obitos_eutanasias",
    TAXA_PROBLEMAS
)


# 18. GERAÇÃO DE DUPLICIDADES DE ANIMAIS

quantidade_duplicados = math.ceil(
    len(animais) * TAXA_DUPLICADOS
)

duplicados_exatos = []
duplicados_parciais = []

animais_selecionados = random.sample(
    animais,
    min(quantidade_duplicados, len(animais))
)

for animal in animais_selecionados:

    copia = deepcopy(animal)

    if random.random() < 0.5:

        # Duplicidade exata:
        # todos os campos permanecem iguais.

        duplicados_exatos.append(copia)

    else:

        # Duplicidade parcial:
        # mantém o identificador de origem,
        # mas altera um campo.

        campo = random.choice([
            "nome",
            "raca",
            "peso_kg",
            "municipio",
            "sexo"
        ])

        if campo == "nome":
            copia["nome"] = "Nome divergente"

        elif campo == "raca":
            copia["raca"] = "Raça divergente"

        elif campo == "peso_kg":
            copia["peso_kg"] = round(
                random.uniform(1, 50),
                2
            )

        elif campo == "municipio":

            municipios_diferentes = [
                municipio
                for municipio in municipios
                if municipio != animal["municipio"]
            ]

            copia["municipio"] = random.choice(
                municipios_diferentes
            )

        elif campo == "sexo":

            copia["sexo"] = (
                "Fêmea"
                if animal["sexo"] == "Macho"
                else "Macho"
            )

        registrar_problema(
            copia,
            "duplicidade_parcial"
        )

        duplicados_parciais.append(copia)


# Adiciona as duplicidades aos animais originais.

animais.extend(duplicados_exatos)
animais.extend(duplicados_parciais)


# 19. ORGANIZAÇÃO DOS CONJUNTOS

conjuntos = {
    "animais": animais,
    "vacinacoes": vacinacoes,
    "diagnosticos": diagnosticos,
    "maus_tratos": maus_tratos,
    "obitos_eutanasias": obitos_eutanasias
}


# 20. PREPARAÇÃO DOS REGISTROS BRUTOS

registros_para_inserir = []

for nome_conjunto, registros in conjuntos.items():

    for registro in registros:

        dados_json = json.dumps(
            registro,
            ensure_ascii=False,
            allow_nan=False
        )

        registros_para_inserir.append((
            nome_conjunto,
            dados_json
        ))


# 21. GRAVAÇÃO TRANSACIONAL NO SQLITE

try:

    cursor.execute("BEGIN IMMEDIATE")

    # Atualiza a sequência para a próxima geração.
    cursor.execute("""
        UPDATE controle_sequencias
        SET proximo_valor = ?
        WHERE nome = 'animais'
    """, (
        proximo_numero + QUANTIDADE_ANIMAIS,
    ))

    # ------------------------------------------------------------
    # CRIAÇÃO DO LOTE
    # ------------------------------------------------------------

    colunas_lotes = {
        linha[1]
        for linha in cursor.execute(
            "PRAGMA table_info(lotes)"
        ).fetchall()
    }

    dados_lote = {
        "tipo_lote": "Geração sintética",
        "quantidade_informada": QUANTIDADE_ANIMAIS,
        "quantidade_gerada": len(animais),
        "data_criacao": datetime.now().isoformat(
            timespec="seconds"
        ),
        "descricao": (
            "Dados sintéticos de zoonoses. "
            "Inclui problemas controlados e duplicidades."
        ),
        "status": "Concluído"
    }

    campos_lote = [
        campo
        for campo in dados_lote
        if campo in colunas_lotes
    ]

    if not campos_lote:
        raise Exception(
            "Não foi possível identificar os campos da tabela lotes."
        )

    sql_lote = (
        "INSERT INTO lotes "
        f"({', '.join(campos_lote)}) "
        f"VALUES ({', '.join(['?'] * len(campos_lote))})"
    )

    cursor.execute(
        sql_lote,
        [
            dados_lote[campo]
            for campo in campos_lote
        ]
    )

    id_lote = cursor.lastrowid


    # ------------------------------------------------------------
    # INSERÇÃO DOS REGISTROS BRUTOS
    # ------------------------------------------------------------

    data_recebimento = datetime.now().isoformat(
        timespec="seconds"
    )

    cursor.executemany("""
        INSERT INTO registros_brutos
        (
            id_lote,
            conjunto,
            dados_json,
            data_recebimento
        )
        VALUES (?, ?, ?, ?)
    """, [
        (
            id_lote,
            conjunto,
            dados_json,
            data_recebimento
        )
        for conjunto, dados_json in registros_para_inserir
    ])


    # ------------------------------------------------------------
    # REGISTRO NA AUDITORIA
    # ------------------------------------------------------------

    colunas_auditoria = {
        linha[1]
        for linha in cursor.execute(
            "PRAGMA table_info(auditoria)"
        ).fetchall()
    }

    dados_auditoria = {
        "entidade": "lotes",
        "id_registro": str(id_lote),
        "operacao": "GERAÇÃO",
        "descricao": (
            f"Lote {id_lote}: "
            f"{len(animais)} registros de animais "
            f"e {len(registros_para_inserir)} registros brutos."
        ),
        "data_operacao": datetime.now().isoformat(
            timespec="seconds"
        )
    }

    campos_auditoria = [
        campo
        for campo in dados_auditoria
        if campo in colunas_auditoria
    ]

    if campos_auditoria:

        sql_auditoria = (
            "INSERT INTO auditoria "
            f"({', '.join(campos_auditoria)}) "
            f"VALUES ({', '.join(['?'] * len(campos_auditoria))})"
        )

        cursor.execute(
            sql_auditoria,
            [
                dados_auditoria[campo]
                for campo in campos_auditoria
            ]
        )


    # Confirma todas as operações.
    conexao.commit()


except Exception:

    conexao.rollback()
    conexao.close()

    raise


# 22. RELATÓRIO DA GERAÇÃO

print("\n" + "=" * 60)
print("GERAÇÃO CONCLUÍDA")
print("=" * 60)

print(f"\nLote gerado: {id_lote}")
print(f"Modo de geração: {MODO_GERACAO}")

print("\nQUANTIDADE DE REGISTROS POR CONJUNTO")

for nome, registros in conjuntos.items():

    print(
        f"{nome}: {len(registros)}"
    )


print("\nPROBLEMAS INTRODUZIDOS NOS REGISTROS ORIGINAIS")

for nome, quantidade in relatorio_problemas.items():

    if nome == "animais":

        total_original = (
            len(animais)
            - len(duplicados_exatos)
            - len(duplicados_parciais)
        )

    else:

        total_original = len(conjuntos[nome])

    percentual = (
        quantidade / total_original * 100
        if total_original > 0
        else 0
    )

    print(
        f"{nome}: {quantidade} registros "
        f"({percentual:.2f}%)"
    )


print("\nDUPLICIDADES DE ANIMAIS")

print(
    f"Duplicidades exatas: {len(duplicados_exatos)}"
)

print(
    f"Duplicidades parciais: {len(duplicados_parciais)}"
)

print(
    f"Total adicional: {quantidade_duplicados}"
)


print("\nTOTAL DE REGISTROS BRUTOS INSERIDOS")

print(
    len(registros_para_inserir)
)

print("\nPRÓXIMO IDENTIFICADOR DISPONÍVEL")

print(
    f"ANM{proximo_numero + QUANTIDADE_ANIMAIS:08d}"
)

print("\n" + "=" * 60)

conexao.close()

print("Banco SQLite atualizado com sucesso.")
print("Os dados anteriores foram preservados.")


# 23. CONFERÊNCIA DAS QUANTIDADES NO BANCO

conexao = sqlite3.connect(CAMINHO_BANCO)
cursor = conexao.cursor()

tabelas_conferencia = [
    "registros_brutos",
    "lotes",
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias",
    "auditoria"
]

print("\nCONFERÊNCIA DAS TABELAS DO BANCO")
print("=" * 60)

for tabela in tabelas_conferencia:

    quantidade = cursor.execute(
        f"SELECT COUNT(*) FROM {tabela}"
    ).fetchone()[0]

    print(
        f"{tabela}: {quantidade} registros"
    )

conexao.close()

print("=" * 60)
